In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Train test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)
# df.iloc[:, 1:] => : means "all rows", 1: means "columns from position 1 to the end" (everything except the first column). This returns a DataFrame.
# df.iloc[:, 0] => : means "all rows", 0 means "column at position 0" (the first column). This returns a Series — the first column's values.

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

"""
These lines standardize the features (mean = 0, std = 1), which most ML models (including neural nets) train better with.
scaler = StandardScaler() → creates a scaler object from scikit-learn that will learn how to transform data: z = (x - mean) / std.

X_train = scaler.fit_transform(X_train) → does two things at once:
fit: computes the mean and standard deviation of each column using only the training data.
transform: applies (x - mean) / std to X_train using those computed values, replacing it with the standardized version.

X_test = scaler.transform(X_test) → applies the same mean/std learned from the training set to the test set — it does not re-fit. 
This is important: the test set must be scaled using training statistics, 
otherwise you'd leak information from the test set into your preprocessing (data leakage), and it also wouldn't reflect how the model would behave on truly unseen data in production.

So the rule of thumb: fit_transform on train, transform (only) on test/validation/new data.
"""

"\nThese lines standardize the features (mean = 0, std = 1), which most ML models (including neural nets) train better with.\nscaler = StandardScaler() → creates a scaler object from scikit-learn that will learn how to transform data: z = (x - mean) / std.\n\nX_train = scaler.fit_transform(X_train) → does two things at once:\nfit: computes the mean and standard deviation of each column using only the training data.\ntransform: applies (x - mean) / std to X_train using those computed values, replacing it with the standardized version.\n\nX_test = scaler.transform(X_test) → applies the same mean/std learned from the training set to the test set — it does not re-fit. \nThis is important: the test set must be scaled using training statistics, \notherwise you'd leak information from the test set into your preprocessing (data leakage), and it also wouldn't reflect how the model would behave on truly unseen data in production.\n\nSo the rule of thumb: fit_transform on train, transform (only) 

In [8]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

"""
Same logic as the StandardScaler. fit_transform makes the labeling of y train and transform the labels inplace
transform on test set uses the labeling done by fit on the train data and transform the test set accordingly.
"""

'\nSame logic as the StandardScaler. fit_transform makes the labeling of y train and transform the labels inplace\ntransform on test set uses the labeling done by fit on the train data and transform the test set accordingly.\n'

In [9]:
y_train

array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
       0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0,

Numpy arrays to Pytorch tensors

In [10]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [11]:
X_train_tensor.shape

torch.Size([455, 30])

In [12]:
# Custom Dataset
from torch.utils.data import Dataset, DataLoader

class CustomDataSet(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [13]:
train_dataset = CustomDataSet(X_train_tensor, y_train_tensor)
test_dataset = CustomDataSet(X_test_tensor, y_test_tensor)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

Defining The Model

In [15]:
class MySimpleNN(nn.Module): 
    def __init__(self, num_of_features):
        super().__init__()
        self.layer = nn.Linear(num_of_features, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        out = self.layer(features)
        out = self.sigmoid(out)
        
        return out
    

In [16]:
learning_rate = 0.1 # the alpha - size of the steps the optimizer takes when updating the NN's weights to minimize errors
epochs = 25 # each epoch is one complete pass of the entire training dataset through the neural network. 
#Because a network cannnot find all underlying patterns in a single pass, it requires multiple epochs to analyze and refine its weights

In [17]:
loss_function = nn.BCELoss()

Training Pipeline

#### The torch.optim Module
`torch.optim` is a module in Pytorch that provides a variety of optimization algorithms used to update the parameters of our model during training.
It includes common optimizers like Stochastic Gradient Descent (SGD), Adam, RMSprop, and many more.
It handles weight updates efficiently, including additional features like learning rate scheduling and weight decay (regularization)

##### The model.parameters()
The `model.parameters()` method in PyTorch retrieves an iterator over all the trainable parameters (weights and biases) in a model. These parameters are instances of `torch.nn.Parameter` and include:
- **Weights:** The weight matrices of layers like nn.Linear, nn.Conv2d, etc.
- **Biases:** The bias terms of layers (if they exist).

The optimizer uses these parameters to compute gradients and update them during training

In [24]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# training model in a loop
for epoch in range(epochs):    
    for batch_features, batch_labels in train_loader:
        
        # forward pass
        y_pred = model(batch_features)
        print(y_pred)

        # loss calculate
        loss = loss_function(y_pred, batch_labels.view(-1, 1))
        # print(f"Epoch: {epoch}, loss: {loss}")

        # clearing gradients, if any, before backward pass is a good strategy
        optimizer.zero_grad()

        # backward pass
        loss.backward()
        
        # Using optim method - parameter update
        optimizer.step()
        
        # print loss in each epoch
        print(f"Loss: {loss.item()}, Epoch: {epoch + 1}")
        
    

tensor([[0.5837],
        [0.4405],
        [0.5435],
        [0.3241],
        [0.5310],
        [0.4855],
        [0.6275],
        [0.6264],
        [0.5185],
        [0.6053],
        [0.3882],
        [0.5861],
        [0.4817],
        [0.5004],
        [0.5266],
        [0.4705],
        [0.5365],
        [0.4429],
        [0.6320],
        [0.5048],
        [0.4721],
        [0.3656],
        [0.3139],
        [0.5341],
        [0.3627],
        [0.5480],
        [0.6142],
        [0.6104],
        [0.5672],
        [0.5595],
        [0.5002],
        [0.4627]], grad_fn=<SigmoidBackward0>)
Loss: 0.6989942789077759, Epoch: 1
tensor([[0.4088],
        [0.3793],
        [0.5047],
        [0.7489],
        [0.5073],
        [0.5250],
        [0.3500],
        [0.6628],
        [0.5835],
        [0.7184],
        [0.6018],
        [0.5447],
        [0.4446],
        [0.3763],
        [0.8524],
        [0.6042],
        [0.4619],
        [0.2822],
        [0.4008],
        [0.4711],


In [25]:
print(f"Weights: {model.layer.weight}")
print(f"Bias: {model.layer.bias}")

Weights: Parameter containing:
tensor([[ 0.6125,  0.4081,  0.2922,  0.4120,  0.1989, -0.0096,  0.4746,  0.6473,
          0.0172, -0.3129,  0.5601, -0.0201,  0.4864,  0.6377,  0.1003, -0.3280,
         -0.0658,  0.0506, -0.1985, -0.3676,  0.7740,  0.7734,  0.4458,  0.5906,
          0.4710,  0.3100,  0.4649,  0.5916,  0.5530,  0.0798]],
       requires_grad=True)
Bias: Parameter containing:
tensor([-0.2968], requires_grad=True)


In [26]:
# model evaluation with new weight and bias value
model.eval() # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()
        
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)
    
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Accuracy: {overall_accuracy:.4f}")

Accuracy: 0.9766
